# nb_07_e2e_verify — assert the pipeline ingested correctly

Read-only **PASS/FAIL** verification for the Sprint 10 E2E test. It compares the Azure AI Search
index against the pipeline's Delta tables and the generator **manifest** (ground truth), scoped
to the generated corpus (files under any `.../e2e/...` path) so it is independent of any other
content in the bucket. Four check groups:

- **A. Reconciliation** — index chunk count == `ingestion_state.chunk_count`, per file and in
  aggregate; no orphans in either direction.
- **B. Page coverage / quality** — every source page is represented: distinct `page_number` in the
  index == manifest page count == `ingestion_log.pages`, and page *N*'s chunk actually contains
  the `[pN` text marker (right content on the right page).
- **C. Security trimming** — a filtered query per group returns exactly the files that group may
  see; a non-member group sees nothing; `engineering/*` (no ACL) never appears.
- **D. Incremental** *(PHASE='incremental')* — only the mutated files were reprocessed; the deleted
  file has 0 chunks; the modified PDF's page count grew; untouched files were not touched.

## Inputs
- `PHASE` parameter: `baseline` (after first 01->03) or `incremental` (after mutate + 01->03).
- Manifest at `Files/e2e/manifest.json` (uploaded by `run_e2e_test.ps1`); for `incremental` also
  `Files/e2e/manifest_after.json`.
- `config` keys + Key Vault Search key (same as nb_03/05).

## Outputs
- Prints each assertion with `PASS`/`FAIL`, then a final `N passed / M failed` line.
- Writes a machine-readable summary to `Files/_diag/e2e_result.json` for headless runs.
- **Raises** at the end if anything failed (so a job run shows as Failed).


## Parameters


In [ ]:
PHASE = 'baseline'  # 'baseline' or 'incremental'


## Config + Search auth
Search admin key from Key Vault (Search token audience isn't issuable in Fabric).


In [ ]:
import requests, notebookutils, json
from pyspark.sql import functions as F
cfg = {r['key']: r['value'] for r in spark.table('config').collect()}
SEARCH_ENDPOINT = cfg['search_endpoint'].rstrip('/')
INDEX_NAME = cfg['search_index_name']
SEARCH_API = '2024-07-01'
VAULT_URL = f"https://{cfg['kv_name']}.vault.azure.net/"
SEARCH_KEY = notebookutils.credentials.getSecret(VAULT_URL, cfg['search_key_secret'])
HDRS = {'api-key': SEARCH_KEY, 'Content-Type': 'application/json'}
E2E_MARK = '/e2e/'  # only verify the generated corpus
print('phase:', PHASE)


## Search helpers
`search_docs` pages through results with a filter; `count_filter` returns a document count for a
filter (used for the security-trimming checks).


In [ ]:
def search_docs(filter=None, select='chunk_id,file_path,page_number,content,allowed_groups',
                search='*', top=1000):
    url = f'{SEARCH_ENDPOINT}/indexes/{INDEX_NAME}/docs/search?api-version={SEARCH_API}'
    out, skip = [], 0
    while True:
        body = {'search': search, 'select': select, 'top': top, 'skip': skip}
        if filter:
            body['filter'] = filter
        r = requests.post(url, headers=HDRS, json=body, timeout=(10, 60))
        r.raise_for_status()
        page = r.json().get('value', [])
        out += page
        if len(page) < top:
            break
        skip += top
    return out

def e2e_only(docs):
    return [d for d in docs if E2E_MARK in (d.get('file_path') or '')]


## Load manifest (ground truth)
The generator manifest describes every generated file: rel_path, folder, ext, expected pages, and
ACL groups. For the incremental phase we also load the mutation manifest and fold its deltas into
the expected sets (remove the deleted file, add the new one, bump the modified PDF's pages).


In [ ]:
man = json.loads(spark.read.text('Files/e2e/manifest.json', wholetext=True).collect()[0][0])
gen = [g for g in man['generated'] if g['ingested']]  # engineering/* (no_acl) excluded
# expected pages per rel_path (pdf/txt only; docx pages are DI-decided -> None)
exp_pages = {g['rel_path']: g['pages'] for g in gen}
exp_groups = {g['rel_path']: set(g['groups']) for g in gen}
DEL_REL = MOD_PDF_REL = None
if PHASE == 'incremental':
    aft = json.loads(spark.read.text('Files/e2e/manifest_after.json', wholetext=True)
                     .collect()[0][0])['changes']
    for a in aft['added']:
        exp_pages[a['rel_path']] = a['pages_after']; exp_groups[a['rel_path']] = set(a['groups'])
    for m in aft['modified']:
        exp_pages[m['rel_path']] = m['pages_after']; exp_groups[m['rel_path']] = set(m['groups'])
        if m['ext'] == 'pdf':
            MOD_PDF_REL = m['rel_path']
    for d in aft['deleted']:
        DEL_REL = d['rel_path']
        exp_pages.pop(d['rel_path'], None); exp_groups.pop(d['rel_path'], None)
print(f'expected ingested generated files: {len(exp_pages)}')


## Pull the generated corpus from the index once
One paged query, then post-filter to `/e2e/` chunks. Everything downstream reconciles against
this in-memory snapshot + the Delta tables. `content_vector` is intentionally not selected.


In [ ]:
all_docs = e2e_only(search_docs())
by_file = {}
for d in all_docs:
    by_file.setdefault(d['file_path'], []).append(d)
# index file_path is the full abfss URL; map manifest rel_path -> matching index file_path
def match_fp(rel):
    hits = [fp for fp in by_file if fp.endswith(rel)]
    return hits[0] if hits else None
print(f'generated chunks in index: {len(all_docs)} across {len(by_file)} files')


## Result collector


In [ ]:
RESULTS = []
def check(name, ok, detail=''):
    RESULTS.append((name, bool(ok), detail))
    print(f"  [{'PASS' if ok else 'FAIL'}] {name}" + (f'  -- {detail}' if detail else ''))


## A. Reconciliation — index vs `ingestion_state`
Per generated file the index chunk count must equal `ingestion_state.chunk_count`; the aggregate
must match; and neither side may have orphans (a file in one but not the other).


In [ ]:
print('A. Reconciliation')
st = (spark.table('ingestion_state')
      .where(F.col('file_path').contains(E2E_MARK))
      .select('file_path', 'chunk_count').collect())
state_counts = {r['file_path']: r['chunk_count'] for r in st}
idx_counts = {fp: len(ch) for fp, ch in by_file.items()}

state_sum = sum(state_counts.values())
idx_sum = sum(idx_counts.values())
check('aggregate chunk count (index == ingestion_state)', state_sum == idx_sum,
      f'index={idx_sum} state={state_sum}')

orphan_idx = sorted(set(idx_counts) - set(state_counts))
orphan_state = sorted(set(state_counts) - set(idx_counts))
check('no index files missing from ingestion_state', not orphan_idx, str(orphan_idx[:3]))
check('no ingestion_state files missing from index', not orphan_state, str(orphan_state[:3]))

mism = [(fp, idx_counts[fp], state_counts.get(fp)) for fp in idx_counts
        if state_counts.get(fp) != idx_counts[fp]]
check('per-file chunk counts match', not mism, f'{len(mism)} mismatched: {mism[:3]}')


## B. Page coverage / quality
For every expected PDF/txt file: the set of distinct `page_number` values in the index must equal
the manifest page count AND `ingestion_log.pages`; and page *N*'s chunk must contain the `[pN`
marker the generator wrote (proving the right page's text landed on the right page number).


In [ ]:
print('B. Page coverage / quality')
# latest ingestion_log pages per file (max ts_utc row per file_path)
# use a window (row_number) instead of a self-join to avoid ambiguous-column errors
from pyspark.sql.window import Window
il = spark.table('ingestion_log').where(F.col('file_path').contains(E2E_MARK))
_wspec = Window.partitionBy('file_path').orderBy(F.col('ts_utc').desc())
il_latest = (il.withColumn('_rn', F.row_number().over(_wspec))
             .where(F.col('_rn') == 1)
             .select('file_path', 'pages').collect())
log_pages = {r['file_path']: r['pages'] for r in il_latest}

missing_pages, marker_fail, logmismatch = [], [], []
for rel, pages in exp_pages.items():
    if pages is None:
        continue  # docx: page count is DI-decided, skip strict check
    fp = match_fp(rel)
    if not fp:
        missing_pages.append(rel); continue
    idx_pages = sorted({d['page_number'] for d in by_file[fp]})
    if idx_pages != list(range(1, pages + 1)):
        missing_pages.append((rel, idx_pages, pages)); continue
    if log_pages.get(fp) != pages:
        logmismatch.append((rel, log_pages.get(fp), pages))
    # The [pN marker is only injected into generated PDFs (multipage_pdf); plain .txt/.md
    # notes have no page markers, so scope this 'right text on right page' check to PDFs.
    if rel.lower().endswith('.pdf'):
        for d in by_file[fp]:
            marker = f"[p{d['page_number']}"
            if marker not in (d.get('content') or ''):
                marker_fail.append((rel, d['page_number'])); break
check('every PDF/txt page present in index (1..N)', not missing_pages,
      f'{len(missing_pages)} bad: {missing_pages[:3]}')
check('index page count == ingestion_log.pages', not logmismatch,
      f'{len(logmismatch)} bad: {logmismatch[:3]}')
check('page N chunk contains the [pN marker (PDF)', not marker_fail,
      f'{len(marker_fail)} bad: {marker_fail[:3]}')


## C. Security trimming
For each group we issue a **real filtered Search query** (the exact filter the calling app uses
after resolving a user's Entra groups) and confirm the returned generated files are exactly those
the manifest says that group may see. A non-member group (`999...`) must see nothing, and
`engineering/*` must never appear for any group.


In [ ]:
print('C. Security trimming')
GROUPS = {
    'g111': '11111111-1111-1111-1111-111111111111',
    'g222': '22222222-2222-2222-2222-222222222222',
    'g333': '33333333-3333-3333-3333-333333333333',
    'g444': '44444444-4444-4444-4444-444444444444',
    'g999': '99999999-9999-9999-9999-999999999999',
}
def expected_for(guid):
    return {rel for rel, gs in exp_groups.items() if guid in gs}

for label, guid in GROUPS.items():
    filt = f"allowed_groups/any(g: search.in(g, '{guid}'))"
    got_fps = {d['file_path'] for d in e2e_only(search_docs(filter=filt, select='file_path'))}
    got = {rel for rel in exp_pages if match_fp(rel) in got_fps}
    # include any returned generated file even if not in exp (to catch leaks)
    leaked = got_fps - {match_fp(rel) for rel in exp_pages}
    exp = expected_for(guid)
    check(f'trimming {label} returns exactly its files', got == exp and not leaked,
          f'got={len(got)} exp={len(exp)} leaked={len(leaked)}')

eng_leak = [d['file_path'] for d in all_docs if '/engineering/' in d['file_path']]
check('engineering/* never indexed (no_acl skip)', not eng_leak, str(eng_leak[:3]))


## D. Incremental correctness
Only runs when `PHASE='incremental'`. Proves the change-detection story: exactly the mutated
files were reprocessed in the latest run, the deleted file has no chunks, and the modified PDF's
page count grew.


In [ ]:
if PHASE != 'incremental':
    print('D. Incremental — skipped (PHASE=baseline)')
else:
    print('D. Incremental correctness')
    aft = json.loads(spark.read.text('Files/e2e/manifest_after.json', wholetext=True)
                     .collect()[0][0])['changes']
    changed_rel = {c['rel_path'] for c in aft['modified'] + aft['added']}
    # latest run_id by max ts_utc
    lr = (spark.table('ingestion_log').orderBy(F.col('ts_utc').desc()).limit(1)
          .collect()[0]['run_id'])
    run2 = {r['file_path'] for r in spark.table('ingestion_log')
            .where((F.col('run_id') == lr) & F.col('file_path').contains(E2E_MARK))
            .select('file_path').collect()}
    run2_rel = {rel for rel in changed_rel if match_fp(rel) in run2}
    # only the modified+added files should be in the latest run
    only_changed = all(any(fp.endswith(rel) for rel in changed_rel) for fp in run2)
    check('latest run reprocessed only modified+added files',
          only_changed and run2_rel == changed_rel,
          f'run2={len(run2)} changed={len(changed_rel)}')
    # deleted file: 0 chunks in index + purged (state row gone, status_reason deleted_purged)
    del_fp = match_fp(DEL_REL) if DEL_REL else None
    check('deleted file has 0 chunks in index', del_fp is None,
          f'still present: {del_fp}')
    dstate = spark.table('ingestion_state').where(F.col('file_path').endswith(DEL_REL)).count()
    check('deleted file removed from ingestion_state', dstate == 0, f'rows={dstate}')
    drow = (spark.table('file_metadata')
            .where(F.col('file_path').endswith(DEL_REL))
            .select('process_status', 'status_reason').collect())
    check('deleted file purged (status_reason == deleted_purged)',
          bool(drow) and drow[0]['status_reason'] == 'deleted_purged',
          str([(r['process_status'], r['status_reason']) for r in drow]))
    # modified PDF grew to its new page count
    if MOD_PDF_REL:
        fp = match_fp(MOD_PDF_REL)
        pgs = sorted({d['page_number'] for d in by_file.get(fp, [])}) if fp else []
        check('modified PDF now has the new page count',
              pgs == list(range(1, exp_pages[MOD_PDF_REL] + 1)),
              f'pages={pgs} exp={exp_pages[MOD_PDF_REL]}')


## Summary + machine-readable result
Prints `N passed / M failed`, writes `Files/_diag/e2e_result.json` (for headless retrieval via
`read_onelake_json.ps1`), and raises if anything failed so a job run reports Failed.


In [ ]:
passed = sum(1 for _, ok, _ in RESULTS if ok)
failed = [n for n, ok, _ in RESULTS if not ok]
print(f'\n=== {passed} passed / {len(failed)} failed  (phase={PHASE}) ===')
for n in failed:
    print('  FAILED:', n)
summary = {'phase': PHASE, 'passed': passed, 'failed': len(failed),
           'failed_names': failed,
           'results': [{'name': n, 'ok': ok, 'detail': d} for n, ok, d in RESULTS]}
try:
    notebookutils.fs.put('Files/_diag/e2e_result.json', json.dumps(summary, indent=2),
                         overwrite=True)
    print('wrote Files/_diag/e2e_result.json')
except Exception as e:
    print('could not write result json:', e)
if failed:
    raise SystemExit(f'{len(failed)} E2E assertion(s) FAILED: {failed}')
print('ALL E2E ASSERTIONS PASSED')
